# Chapter 17 — The Lebesgue Integral

Companion notebook to Chapter 17 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Integrals over an interval are computed by the midpoint rule on a fine grid,
which is deterministic, or exactly with fractions. Random samples are used
only in the sections on gradient estimators, where a cell says so.

In [1]:
import math
import warnings
from fractions import Fraction

import torch
from torch.func import grad, vmap

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def midpoints(a, b, n):
    # Midpoints of n equal cells of [a, b] and the cell length.
    h = (b - a) / n
    return a + h * (torch.arange(n) + 0.5), h


def integral(f, a, b, n=2_000_000):
    x, h = midpoints(a, b, n)
    return (f(x) * h).sum()

2.15.0+cu132


## 1. One simple function, three measures (Definition 17.1, Exercise 17.1)

The integral of $s = \sum_i a_i 1_{A_i}$ is $\sum_i a_i\,\mu(A_i)$. The
function is fixed; the measure decides how the sets $A_i$ are weighed.

In [2]:
# f = 2 on [0, 1), 5 on [1, 3).
values = [(2, (0, 1)), (5, (1, 3))]


def lebesgue(a, b):
    return b - a


def dirac_mix(a, b):                 # delta_1 + 2 delta_2
    return sum(w for p, w in [(1, 1), (2, 2)] if a <= p < b)


sample = [Fraction(1, 2), Fraction(3, 2), Fraction(5, 2), Fraction(7)]


def empirical(a, b):
    return Fraction(sum(1 for p in sample if a <= p < b), len(sample))


results = {}
for name, mu in [("Lebesgue", lebesgue), ("delta_1 + 2 delta_2", dirac_mix),
                 ("empirical", empirical)]:
    results[name] = sum(a * mu(*A) for a, A in values)
    print(f"{name:22s} integral = {results[name]}")

assert results["Lebesgue"] == 12
assert results["delta_1 + 2 delta_2"] == 15
assert results["empirical"] == 3

Lebesgue               integral = 12
delta_1 + 2 delta_2    integral = 15
empirical              integral = 3


## 2. Cutting the range (Lemma 17.4, Exercise 17.2)

$s_n = \min(n, 2^{-n}\lfloor 2^n f\rfloor)$ rounds $f$ down to a multiple of
$2^{-n}$. For $f(x) = x$ on $[0, 1)$ its integral is
$\frac12(1 - 2^{-n})$, computed here exactly.

In [3]:
for n in range(1, 9):
    N = 2**n
    # s_n = k / N on [k / N, (k + 1) / N), a set of Lebesgue measure 1 / N.
    I = sum(Fraction(k, N) * Fraction(1, N) for k in range(N))
    assert I == Fraction(1, 2) * (1 - Fraction(1, N))
    print(f"n = {n}:  integral of s_n = {I} = {float(I):.6f}")

n = 1:  integral of s_n = 1/4 = 0.250000
n = 2:  integral of s_n = 3/8 = 0.375000
n = 3:  integral of s_n = 7/16 = 0.437500
n = 4:  integral of s_n = 15/32 = 0.468750
n = 5:  integral of s_n = 31/64 = 0.484375
n = 6:  integral of s_n = 63/128 = 0.492188
n = 7:  integral of s_n = 127/256 = 0.496094
n = 8:  integral of s_n = 255/512 = 0.498047


The same lemma for the function of Figure 17.1: the $s_n$ increase, stay
below $f$, and are within $2^{-n}$ of it once $n\geq f$.

In [4]:
def f_fig(x):
    return 0.2 + 1.4 * torch.sin(torch.deg2rad(60 * x))**2 * torch.exp(-0.15 * x)


def s(n, fx):
    return torch.clamp(torch.floor(2.0**n * fx) / 2.0**n, max=float(n))


x, h = midpoints(0.0, 5.0, 1_000_000)
fx = f_fig(x)
exact = (fx * h).sum()
prev = torch.zeros_like(fx)
for n in range(1, 11):
    sn = s(n, fx)
    assert (sn >= prev).all() and (sn <= fx).all()
    if n >= fx.max():
        assert (fx - sn).max() <= 2.0**-n
    prev = sn
    print(f"n = {n:2d}:  integral of s_n = {(sn * h).sum():.6f}"
          f"   max(f - s_n) = {(fx - sn).max():.6f}")
print(f"integral of f       = {exact:.6f}")
assert exact - (prev * h).sum() < 5 * 2.0**-10

n =  1:  integral of s_n = 2.207695   max(f - s_n) = 0.500000
n =  2:  integral of s_n = 2.935791   max(f - s_n) = 0.250000
n =  3:  integral of s_n = 3.256403   max(f - s_n) = 0.125000
n =  4:  integral of s_n = 3.418085   max(f - s_n) = 0.062500
n =  5:  integral of s_n = 3.492385   max(f - s_n) = 0.031250
n =  6:  integral of s_n = 3.529259   max(f - s_n) = 0.015625
n =  7:  integral of s_n = 3.549428   max(f - s_n) = 0.007812
n =  8:  integral of s_n = 3.559125   max(f - s_n) = 0.003906
n =  9:  integral of s_n = 3.564001   max(f - s_n) = 0.001953
n = 10:  integral of s_n = 3.566438   max(f - s_n) = 0.000977
integral of f       = 3.568877


An integral "by levels": the integral of $s_n$ is
$\sum_k k2^{-n}\,\lambda\{k2^{-n}\leq f < (k+1)2^{-n}\}$. Only the measures
of the level sets of $f$ are used, as in the right-hand picture of
Figure 16.1.

In [5]:
n = 8
levels = torch.floor(2.0**n * fx).long()             # k for each grid point
measure = torch.bincount(levels) * h                 # lambda{ f in level k }
k = torch.arange(len(measure))
by_levels = (k / 2.0**n * measure).sum()
print(f"sum over levels  = {by_levels:.6f}")
print(f"integral of s_8  = {(s(8, fx) * h).sum():.6f}")
assert torch.isclose(by_levels, (s(8, fx) * h).sum())
assert torch.isclose(measure.sum(), T(5.0))          # the level sets partition [0, 5]

sum over levels  = 3.559125
integral of s_8  = 3.559125


## 3. Monotone convergence and a series (Theorem 17.5, Exercise 17.3)

$\dfrac{x}{e^x - 1} = \sum_{n\geq1} x e^{-nx}$ with non-negative terms, and
$\int_0^\infty x e^{-nx}\,dx = 1/n^2$. The integrals of the partial sums
increase to the integral of the sum, $\pi^2/6$.

In [6]:
x, h = midpoints(0.0, 60.0, 3_000_000)
full = (x / torch.expm1(x) * h).sum()
print(f"integral of x / (e^x - 1) = {full:.10f}")
print(f"pi^2 / 6                  = {math.pi**2 / 6:.10f}")
assert abs(full - math.pi**2 / 6) < 1e-8

x, h = midpoints(0.0, 60.0, 1_000_000)
partial = torch.zeros_like(x)
previous = 0.0
for n in range(1, 1001):
    partial = partial + x * torch.exp(-n * x)
    if n in (1, 2, 5, 10, 100, 1000):
        I = (partial * h).sum().item()
        target = sum(1 / j**2 for j in range(1, n + 1))
        print(f"N = {n:5d}:  integral of partial sum = {I:.8f}"
              f"   sum of 1/n^2 = {target:.8f}")
        assert abs(I - target) < 1e-5 and I > previous
        previous = I

integral of x / (e^x - 1) = 1.6449340668
pi^2 / 6                  = 1.6449340668


N =     1:  integral of partial sum = 1.00000000   sum of 1/n^2 = 1.00000000
N =     2:  integral of partial sum = 1.25000000   sum of 1/n^2 = 1.25000000
N =     5:  integral of partial sum = 1.46361111   sum of 1/n^2 = 1.46361111
N =    10:  integral of partial sum = 1.54976773   sum of 1/n^2 = 1.54976773


N =   100:  integral of partial sum = 1.63498392   sum of 1/n^2 = 1.63498390


N =  1000:  integral of partial sum = 1.64393472   sum of 1/n^2 = 1.64393457


## 4. Three ways to lose mass (pitfall of §17.3, Figure 17.2, Exercise 17.4)

Each sequence has integral $1$ for every $n$ and tends to $0$ at every
point. The integrals are exact (areas of rectangles).

In [7]:
def escape(n, x):         # 1 on [n, n + 1)
    return ((x >= n) & (x < n + 1)).double()


def concentrate(n, x):    # n on (0, 1 / n)
    return n * ((x > 0) & (x < 1 / n)).double()


def spread(n, x):         # 1 / n on [0, n)
    return ((x >= 0) & (x < n)).double() / n


points = T([0.001, 0.3, 1.5, 40.5])
ns = (1, 10, 40, 1000, 100_000)
for name, fn, area in [("escape", escape, lambda n: 1 * 1),
                       ("concentrate", concentrate, lambda n: n * Fraction(1, n)),
                       ("spread", spread, lambda n: Fraction(1, n) * n)]:
    assert all(area(n) == 1 for n in range(1, 2000))
    vals = torch.stack([fn(n, points) for n in ns])
    print(f"{name:12s} values at {points.tolist()} for n in {ns}:")
    print(vals)
    assert vals[-1].max() <= 1e-5

escape       values at [0.001, 0.3, 1.5, 40.5] for n in (1, 10, 40, 1000, 100000):
tensor([[0., 0., 1., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
concentrate  values at [0.001, 0.3, 1.5, 40.5] for n in (1, 10, 40, 1000, 100000):
tensor([[ 1.,  1.,  0.,  0.],
        [10.,  0.,  0.,  0.],
        [40.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.]])
spread       values at [0.001, 0.3, 1.5, 40.5] for n in (1, 10, 40, 1000, 100000):
tensor([[1.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e-01, 1.0000e-01, 1.0000e-01, 0.0000e+00],
        [2.5000e-02, 2.5000e-02, 2.5000e-02, 0.0000e+00],
        [1.0000e-03, 1.0000e-03, 1.0000e-03, 1.0000e-03],
        [1.0000e-05, 1.0000e-05, 1.0000e-05, 1.0000e-05]])


Fatou's inequality is strict here: $\int\liminf f_n = 0 < 1 =
\liminf\int f_n$. And no integrable function dominates
$f_n = n\,1_{(0, 1/n)}$: the supremum of $f_1,\dots,f_N$ equals $n$ on
$[\frac1{n+1}, \frac1n)$ for $n < N$ and $N$ on $(0, \frac1N)$, so its
integral is the harmonic sum $H_N$, which is unbounded.

In [8]:
def integral_of_sup(N):
    total = Fraction(N) * Fraction(1, N)                      # N on (0, 1/N)
    for n in range(1, N):
        total += n * (Fraction(1, n) - Fraction(1, n + 1))    # n on [1/(n+1), 1/n)
    return total


for N in (1, 10, 100, 1000):
    H = sum(Fraction(1, j) for j in range(1, N + 1))
    assert integral_of_sup(N) == H
    print(f"N = {N:5d}:  integral of sup(f_1..f_N) = H_N = {float(H):.4f}")

# The same supremum computed pointwise on a grid, for N = 50.
x, h = midpoints(0.0, 1.0, 4_000_000)
sup = torch.stack([concentrate(n, x) for n in range(1, 51)]).max(dim=0).values
print(f"grid value for N = 50: {(sup * h).sum():.4f}"
      f"   H_50 = {sum(1 / j for j in range(1, 51)):.4f}")
assert abs((sup * h).sum() - sum(1 / j for j in range(1, 51))) < 1e-3

N =     1:  integral of sup(f_1..f_N) = H_N = 1.0000
N =    10:  integral of sup(f_1..f_N) = H_N = 2.9290
N =   100:  integral of sup(f_1..f_N) = H_N = 5.1874
N =  1000:  integral of sup(f_1..f_N) = H_N = 7.4855


grid value for N = 50: 4.4992   H_50 = 4.4992


## 5. Dominated convergence (Theorem 17.14, Exercise 17.5)

$f_n(x) = (1 - x/n)^n 1_{[0, n]}(x)$ is dominated by $e^{-x}$ and converges
to it, so $\int f_n\to1$. In fact $\int f_n = n/(n + 1)$.

In [9]:
x, h = midpoints(0.0, 200.0, 4_000_000)
g = torch.exp(-x)
for n in (1, 2, 5, 20, 100, 200):
    fn = torch.where(x <= n, (1 - x / n).clamp(min=0)**n, torch.zeros_like(x))
    assert (fn <= g + 1e-15).all()                    # the domination
    I = (fn * h).sum().item()
    L1 = ((fn - g).abs() * h).sum().item()            # integral of |f_n - f|
    print(f"n = {n:4d}:  integral = {I:.6f}   n/(n+1) = {n / (n + 1):.6f}"
          f"   integral of |f_n - f| = {L1:.6f}")
    assert abs(I - n / (n + 1)) < 1e-6
assert L1 < 0.01

n =    1:  integral = 0.500000   n/(n+1) = 0.500000   integral of |f_n - f| = 0.500000
n =    2:  integral = 0.666667   n/(n+1) = 0.666667   integral of |f_n - f| = 0.333333
n =    5:  integral = 0.833333   n/(n+1) = 0.833333   integral of |f_n - f| = 0.166667


n =   20:  integral = 0.952381   n/(n+1) = 0.952381   integral of |f_n - f| = 0.047619
n =  100:  integral = 0.990099   n/(n+1) = 0.990099   integral of |f_n - f| = 0.009901


n =  200:  integral = 0.995025   n/(n+1) = 0.995025   integral of |f_n - f| = 0.004975


## 6. Riemann sums and the absolute integral (Theorem 17.17)

For a continuous function the Riemann sums over cells of the domain and the
sums over levels of the range have the same limit.

In [10]:
def by_domain(f, a, b, N):
    xk = a + (b - a) / N * torch.arange(N)            # left end points
    return (f(xk) * (b - a) / N).sum()


def by_range(f, a, b, n, grid=2_000_000):
    x, h = midpoints(a, b, grid)
    fx = f(x)
    lev = torch.floor(2.0**n * fx)                    # f >= 0 here
    return (lev / 2.0**n * h).sum()


f = lambda x: torch.exp(-x**2)
exact = math.sqrt(math.pi) / 2 * math.erf(2.0)
for N, n in [(10, 3), (100, 6), (1000, 10), (10_000, 13)]:
    d, r = by_domain(f, 0.0, 2.0, N).item(), by_range(f, 0.0, 2.0, n).item()
    print(f"N = {N:6d}: Riemann sum = {d:.6f}    levels 2^-{n:<2d}: {r:.6f}")
print(f"exact value                 = {exact:.6f}")
assert abs(d - exact) < 1e-3 and abs(r - exact) < 1e-3

N =     10: Riemann sum = 0.980007    levels 2^-3 : 0.753713
N =    100: Riemann sum = 0.891896    levels 2^-6 : 0.866106
N =   1000: Riemann sum = 0.883063    levels 2^-10: 0.881098
N =  10000: Riemann sum = 0.882180    levels 2^-13: 0.881959
exact value                 = 0.882081


The Lebesgue integral is an absolute integral. $\int_0^b\frac{\sin x}{x}dx$
converges to $\pi/2$ as $b\to\infty$, but $\int_0^b|\frac{\sin x}{x}|dx$
grows like $\frac2\pi\ln b$: the function is not integrable on
$[0, \infty)$.

In [11]:
sinc = lambda x: torch.sin(x) / x
previous = 0.0
for b in (10.0, 100.0, 1000.0, 10_000.0):
    I = integral(sinc, 0.0, b, 8_000_000).item()
    A = integral(lambda x: sinc(x).abs(), 0.0, b, 8_000_000).item()
    print(f"b = {b:8.0f}:  integral = {I:.5f}   integral of |.| = {A:.4f}"
          f"   (2/pi) ln b = {2 / math.pi * math.log(b):.4f}")
    assert A - previous > 1.4                         # about (2/pi) ln 10 = 1.47
    previous = A
assert abs(I - math.pi / 2) < 1e-3

b =       10:  integral = 1.65835   integral of |.| = 2.5587   (2/pi) ln b = 1.4659


b =      100:  integral = 1.56223   integral of |.| = 4.0467   (2/pi) ln b = 2.9317
b =     1000:  integral = 1.57023   integral of |.| = 5.5104   (2/pi) ln b = 4.3976


b =    10000:  integral = 1.57089   integral of |.| = 6.9764   (2/pi) ln b = 5.8635


## 7. Differentiating under the integral sign (Theorem 17.19, Exercise 17.6)

$F(\theta) = \int_0^\infty e^{-x}\cos(\theta x)\,dx = 1/(1 + \theta^2)$. The
derivative of the integrand is bounded by $xe^{-x}$, which is integrable, so
$F'(\theta) = -\int_0^\infty xe^{-x}\sin(\theta x)\,dx$. Three computations
of $F'$ agree: the closed form, automatic differentiation through the
quadrature (derivative of the integral), and the quadrature of the
derivative (integral of the derivative).

In [12]:
x, h = midpoints(0.0, 60.0, 3_000_000)


def F(theta):
    return (torch.exp(-x) * torch.cos(theta * x) * h).sum()


for th in (0.0, 0.5, 1.0, 3.0):
    theta = T(th)
    closed = -2 * th / (1 + th**2)**2
    d_of_int = grad(F)(theta).item()
    int_of_d = -(x * torch.exp(-x) * torch.sin(theta * x) * h).sum().item()
    print(f"theta = {th}:  F = {F(theta):.8f}   F' closed = {closed:+.8f}"
          f"   d(int) = {d_of_int:+.8f}   int(d) = {int_of_d:+.8f}")
    assert abs(F(theta) - 1 / (1 + th**2)) < 1e-8
    assert abs(d_of_int - closed) < 1e-8 and abs(int_of_d - closed) < 1e-8
    # the bound used in the theorem
    assert (x * torch.exp(-x) * torch.sin(theta * x).abs()
            <= x * torch.exp(-x) + 1e-18).all()
print(f"integral of the bound x e^-x = {(x * torch.exp(-x) * h).sum():.8f}")

theta = 0.0:  F = 1.00000000   F' closed = -0.00000000   d(int) = +0.00000000   int(d) = -0.00000000
theta = 0.5:  F = 0.80000000   F' closed = -0.64000000   d(int) = -0.64000000   int(d) = -0.64000000
theta = 1.0:  F = 0.50000000   F' closed = -0.50000000   d(int) = -0.50000000   int(d) = -0.50000000


theta = 3.0:  F = 0.10000000   F' closed = -0.06000000   d(int) = -0.06000000   int(d) = -0.06000000
integral of the bound x e^-x = 1.00000000


### The pitfall: a derivative that vanishes almost everywhere

$f(x, \theta) = 1_{\{x\leq\theta\}}$ on $[0, 1]$. For every $x\neq\theta$
the derivative in $\theta$ is $0$, and automatic differentiation returns
$0$. But $F(\theta) = \theta$ has derivative $1$. Hypothesis (1) of the
theorem fails: the difference quotient of $f$ at $\theta_0$ equals
$1/|\theta - \theta_0|$ for $x$ between $\theta_0$ and $\theta$. The
smallest function that bounds all these quotients is $1/|x - \theta_0|$,
which is not integrable: its integral over $|x - \theta_0| > \delta$ grows
like $2\ln(1/\delta)$.

In [13]:
x, h = midpoints(0.0, 1.0, 1_000_000)


def F_step(theta):
    # 1_{x <= theta}, written so that the graph depends on theta.
    return (torch.where(x <= theta, 1.0, 0.0) * h + 0 * theta).sum()


theta = T(0.4)
pathwise = grad(F_step)(theta).item()
eps = 1e-3
finite_diff = ((F_step(theta + eps) - F_step(theta - eps)) / (2 * eps)).item()
print(f"F(0.4)                         = {F_step(theta):.6f}")
print(f"integral of the derivative     = {pathwise}")
print(f"derivative of the integral     = {finite_diff:.6f}")
assert pathwise == 0.0 and abs(finite_diff - 1.0) < 1e-6

# The difference quotients have integral 1 but are not dominated:
for d in (0.1, 0.01, 0.001):
    q = (torch.where(x <= theta + d, 1.0, 0.0) - torch.where(x <= theta, 1.0, 0.0)) / d
    print(f"d = {d}:  max of quotient = {q.max():.0f}   integral = {(q * h).sum():.4f}")

# The smallest bound of all the quotients, 1 / |x - theta|, is not integrable.
previous = -math.inf
for delta in (1e-1, 1e-2, 1e-3, 1e-4):
    far = (x - theta).abs() > delta
    I = (h / (x[far] - theta).abs()).sum().item()
    exact = math.log(0.4 / delta) + math.log(0.6 / delta)
    print(f"delta = {delta:.0e}:  integral of 1/|x - theta| outside = {I:.4f}"
          f"   ln(0.4/delta) + ln(0.6/delta) = {exact:.4f}")
    assert abs(I - exact) < 0.02 and I > previous + 4
    previous = I

F(0.4)                         = 0.400000
integral of the derivative     = 0.0
derivative of the integral     = 1.000000
d = 0.1:  max of quotient = 10   integral = 1.0000
d = 0.01:  max of quotient = 100   integral = 1.0000
d = 0.001:  max of quotient = 1000   integral = 1.0000
delta = 1e-01:  integral of 1/|x - theta| outside = 3.1781   ln(0.4/delta) + ln(0.6/delta) = 3.1781
delta = 1e-02:  integral of 1/|x - theta| outside = 7.7832   ln(0.4/delta) + ln(0.6/delta) = 7.7832
delta = 1e-03:  integral of 1/|x - theta| outside = 12.3884   ln(0.4/delta) + ln(0.6/delta) = 12.3884
delta = 1e-04:  integral of 1/|x - theta| outside = 16.9936   ln(0.4/delta) + ln(0.6/delta) = 16.9936


## 8. Integrating against a push-forward (Theorem 17.22, Exercise 17.7)

$\int g\,d(f_\#\mu) = \int g\circ f\,d\mu$. First on a die, exactly: $Z$
uniform on $\{1,\dots,6\}$, $h(x) = x \bmod 3$, $g(y) = y^2$.

In [14]:
P_Z = {z: Fraction(1, 6) for z in range(1, 7)}
h_map = lambda z: z % 3
g = lambda y: y**2

direct = sum(g(h_map(z)) * p for z, p in P_Z.items())      # int g o h dP_Z

law = {}                                                    # h_# P_Z
for z, p in P_Z.items():
    law[h_map(z)] = law.get(h_map(z), 0) + p
via_law = sum(g(y) * p for y, p in law.items())             # int g d(h_# P_Z)

print("law of h(Z):", {k: str(v) for k, v in sorted(law.items())})
print(f"E[h(Z)^2] directly = {direct},  from the law = {via_law}")
assert direct == via_law == Fraction(5, 3)

law of h(Z): {0: '1/3', 1: '1/3', 2: '1/3'}
E[h(Z)^2] directly = 5/3,  from the law = 5/3


Then for a density: $Z$ standard Gaussian and $Y = e^Z$, which is
log-normal with density $\varphi(\ln y)/y$. The mean of $Y$ is computed in
the space of $z$ and in the space of $y$; both give $e^{1/2}$.

In [15]:
phi = lambda z: torch.exp(-z**2 / 2) / math.sqrt(2 * math.pi)

in_z = integral(lambda z: torch.exp(z) * phi(z), -12.0, 14.0).item()
# In y the density has a long tail; integrate on a logarithmic grid.
u, du = midpoints(-12.0, 14.0, 2_000_000)
y = torch.exp(u)
dy = y * du
in_y = (y * phi(torch.log(y)) / y * dy).sum().item()
print(f"int e^z phi(z) dz        = {in_z:.10f}")
print(f"int y dP_Y(y)            = {in_y:.10f}")
print(f"e^(1/2)                  = {math.exp(0.5):.10f}")
assert abs(in_z - math.exp(0.5)) < 1e-9 and abs(in_y - math.exp(0.5)) < 1e-9

int e^z phi(z) dz        = 1.6487212707
int y dP_Y(y)            = 1.6487212707
e^(1/2)                  = 1.6487212707


## 9. Two estimators of the gradient of an expectation (§17.7, Exercise 17.8)

$z = \mu + \varepsilon$ with $\varepsilon$ standard Gaussian and
$\ell(z) = z^2$, so $L(\mu) = \mu^2 + 1$ and $L'(\mu) = 2\mu$.

* Reparametrization: $\frac{d}{d\mu}\ell(\mu + \varepsilon) =
  2(\mu + \varepsilon)$.
* Score function: $\ell(z)\,\frac{d}{d\mu}\ln\rho(z;\mu) =
  z^2 (z - \mu)$.

Their exact means and variances are integrals against the Gaussian density,
computed first by quadrature.

In [16]:
mu = T(1.0)
loss = lambda z: z**2
log_density = lambda m, z: -(z - m)**2 / 2 - 0.5 * math.log(2 * math.pi)

# Both estimators by automatic differentiation, as functions of one sample.
pathwise = vmap(grad(lambda m, e: loss(m + e)), in_dims=(None, 0))
score = vmap(grad(log_density), in_dims=(None, 0))

eps, h = midpoints(-14.0, 14.0, 2_000_000)
w = phi(eps) * h                                   # the Gaussian measure
est_path = pathwise(mu, eps)
est_score = loss(mu + eps) * score(mu, mu + eps)

for name, est, var_exact in [("reparametrization", est_path, 4.0),
                             ("score function", est_score, 30.0)]:
    mean = (est * w).sum().item()
    var = ((est - mean)**2 * w).sum().item()
    print(f"{name:18s} mean = {mean:.8f}   variance = {var:.8f}")
    assert abs(mean - 2.0) < 1e-8 and abs(var - var_exact) < 1e-7

second_moment = ((1 + eps)**4 * eps**2 * w).sum().item()
print(f"E[(1 + eps)^4 eps^2] = {second_moment:.8f}")
assert abs(second_moment - 34.0) < 1e-7

reparametrization  mean = 2.00000000   variance = 4.00000000


score function     mean = 2.00000000   variance = 30.00000000
E[(1 + eps)^4 eps^2] = 34.00000000


The same with random samples (this cell uses a seeded random generator).
With $N$ samples the standard error is $\sqrt{\text{variance}/N}$: the
score-function estimator needs $30/4 = 7.5$ times as many samples.

In [17]:
N = 400_000
e = torch.randn(N)
p_samples = pathwise(mu, e)
s_samples = loss(mu + e) * score(mu, mu + e)
for name, smp, var_exact in [("reparametrization", p_samples, 4.0),
                             ("score function", s_samples, 30.0)]:
    se = math.sqrt(var_exact / N)
    print(f"{name:18s} mean = {smp.mean():.4f}  (standard error {se:.4f})"
          f"   sample variance = {smp.var():.3f}")
    assert abs(smp.mean() - 2.0) < 5 * se
    assert abs(smp.var() - var_exact) < 0.05 * var_exact + 0.5

reparametrization  mean = 1.9976  (standard error 0.0032)   sample variance = 3.986
score function     mean = 1.9865  (standard error 0.0087)   sample variance = 29.828


## 10. A loss with a jump (Figure 17.3, Exercise 17.9)

$\ell(z) = 1_{\{z > 0\}}$, so $L(\mu) = \Phi(\mu)$ and
$L'(\mu) = \varphi(\mu)$. Every sampling path
$\mu\mapsto\ell(\mu + \varepsilon)$ is flat where it is differentiable: the
reparametrization formula returns $0$. The score-function formula returns
$\varphi(\mu)$.

In [18]:
Phi = lambda m: 0.5 * (1 + math.erf(m / math.sqrt(2)))
step = lambda z: torch.where(z > 0, 1.0, 0.0) + 0 * z       # keeps the graph

pathwise_step = vmap(grad(lambda m, e: step(m + e)), in_dims=(None, 0))

for m in (-1.0, 0.0, 0.7):
    mu = T(m)
    path = (pathwise_step(mu, eps) * w).sum().item()
    sc = (step(mu + eps) * score(mu, mu + eps) * w).sum().item()
    d = 1e-4
    fd = (Phi(m + d) - Phi(m - d)) / (2 * d)
    print(f"mu = {m:+.1f}:  reparametrization = {path}   score = {sc:.6f}"
          f"   phi(mu) = {phi(mu).item():.6f}   finite difference = {fd:.6f}")
    assert path == 0.0
    assert abs(sc - phi(mu).item()) < 1e-5 and abs(fd - phi(mu).item()) < 1e-7

mu = -1.0:  reparametrization = 0.0   score = 0.241969   phi(mu) = 0.241971   finite difference = 0.241971
mu = +0.0:  reparametrization = 0.0   score = 0.398942   phi(mu) = 0.398942   finite difference = 0.398942
mu = +0.7:  reparametrization = 0.0   score = 0.312254   phi(mu) = 0.312254   finite difference = 0.312254


A smooth relaxation restores the Lipschitz hypothesis: replace the step by
$\sigma(z/\tau)$. The reparametrization formula then applies, and its value
tends to $\varphi(\mu)$ as $\tau\to0$, at the price of a variance that
grows like $1/\tau$.

In [19]:
mu = T(0.7)
target = phi(mu).item()
previous_error, previous_var = 1.0, 0.0
for tau in (1.0, 0.3, 0.1, 0.03, 0.01):
    relaxed = vmap(grad(lambda m, e: torch.sigmoid((m + e) / tau)),
                   in_dims=(None, 0))
    est = relaxed(mu, eps)
    mean = (est * w).sum().item()
    var = ((est - mean)**2 * w).sum().item()
    print(f"tau = {tau:5.2f}:  mean of pathwise gradient = {mean:.6f}"
          f"   variance = {var:8.4f}   tau * variance = {tau * var:.4f}")
    assert abs(mean - target) < previous_error and var > previous_var
    previous_error, previous_var = abs(mean - target), var
print(f"phi(0.7) = {target:.6f}")
assert previous_error < 1e-4

tau =  1.00:  mean of pathwise gradient = 0.191958   variance =   0.0031   tau * variance = 0.0031


tau =  0.30:  mean of pathwise gradient = 0.290695   variance =   0.0840   tau * variance = 0.0252
tau =  0.10:  mean of pathwise gradient = 0.309654   variance =   0.4228   tau * variance = 0.0423
tau =  0.03:  mean of pathwise gradient = 0.312018   variance =   1.6369   tau * variance = 0.0491
tau =  0.01:  mean of pathwise gradient = 0.312228   variance =   5.1066   tau * variance = 0.0511
phi(0.7) = 0.312254


## 11. A kink is harmless (second ML box of §17.7)

$\ell(x, \theta) = \max(x - \theta, 0)$ with $x$ standard Gaussian. The
loss is Lipschitz in $\theta$ with constant $1$, and differentiable at
$\theta$ for every $x\neq\theta$, a null set. Theorem 17.19 applies:
$R'(\theta) = \mathbb E[\partial_\theta\ell] = -P(x > \theta) =
\Phi(\theta) - 1$.

In [20]:
relu_loss = lambda th, x: torch.relu(x - th)
d_theta = vmap(grad(relu_loss), in_dims=(None, 0))

xs, h = midpoints(-14.0, 14.0, 2_000_000)
wx = phi(xs) * h
R = lambda th: (relu_loss(th, xs) * wx).sum()

for th in (-0.5, 0.0, 1.2):
    theta = T(th)
    mean_grad = (d_theta(theta, xs) * wx).sum().item()     # E[d ell]
    grad_mean = grad(R)(theta).item()                      # d E[ell]
    print(f"theta = {th:+.1f}:  E[d ell] = {mean_grad:.8f}"
          f"   dR = {grad_mean:.8f}   Phi(theta) - 1 = {Phi(th) - 1:.8f}")
    assert abs(mean_grad - (Phi(th) - 1)) < 1e-5
    assert abs(grad_mean - mean_grad) < 1e-12
    # Lipschitz constant 1 in theta
    assert ((relu_loss(theta, xs) - relu_loss(theta + 0.3, xs)).abs()
            <= 0.3 + 1e-12).all()

theta = -0.5:  E[d ell] = -0.69146105   dR = -0.69146105   Phi(theta) - 1 = -0.69146246
theta = +0.0:  E[d ell] = -0.50000000   dR = -0.50000000   Phi(theta) - 1 = -0.50000000


theta = +1.2:  E[d ell] = -0.11507045   dR = -0.11507045   Phi(theta) - 1 = -0.11506967


## Summary

* The integral of a simple function is a finite sum weighted by the measure;
  the same function has integrals 12, 15 and 3 for three measures.
* Rounding $f$ down to multiples of $2^{-n}$ gives integrals that increase
  to $\int f$; for $f(x) = x$ on $[0, 1)$ they are $\frac12(1 - 2^{-n})$.
* $\int_0^\infty x/(e^x - 1)\,dx = \sum 1/n^2 = \pi^2/6$ by monotone
  convergence; $\int_0^n(1 - x/n)^n dx = n/(n+1)\to1$ by dominated
  convergence.
* Mass can escape, concentrate or spread: integral $1$ for all $n$ and
  limit $0$ everywhere. The supremum of $n1_{(0,1/n)}$ has integral $H_N$.
* $F(\theta) = \int_0^\infty e^{-x}\cos\theta x\,dx$ is differentiated under
  the integral sign; $1_{\{x\leq\theta\}}$ is not.
* $\mathbb E[h(Z)^2] = 5/3$ on the die, from $P_Z$ or from the law of $h(Z)$.
* For $\ell(z) = z^2$ at $\mu = 1$ both gradient estimators have mean $2$,
  with variances $4$ and $30$. For a step loss the reparametrization
  formula gives $0$ and the score function gives $\varphi(\mu)$.